In [1]:
from typing import Tuple, Dict
import pandas as pd
import numpy as np
import random

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from pytorch_lightning import LightningModule, Trainer, seed_everything
from torch_geometric.nn import GCNConv
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import os, sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from common import utils
from common.utils import DataPreprocessor, UserItemPairDataset

from dotenv import load_dotenv
load_dotenv(dotenv_path="../.env")
MLFLOW_SERVICE_URI = os.getenv("MLFLOW_SERVICE_URI", "")

MOVIELENS_DATA_DIR = "../datasets/movielens-2k/user_ratedmovies.dat"
RANDOM_SEED = 42
BATCH_SIZE = 1024

utils.set_seed(RANDOM_SEED)


/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/.venv/lib/python3.11/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
Seed set to 42


random seed set to 42
numpy seed set to 42
torch seed set to 42
lightning seed set to 42


In [2]:
interaction_df = DataPreprocessor().load_and_process_df(
    file_dir=MOVIELENS_DATA_DIR,
    year_range=(2006, 2008),
)
NUM_USER = interaction_df["userID"].nunique()
NUM_ITEM = interaction_df["movieID"].nunique()
interaction_df.head()

Data count: 855598
Data count after filtering by year (2006, 2008): 480608
Num of distinct users: 2103
Num of distinct items: 9519
done!
------------------------------
Filtering by min user/item interactions (0/0):
Data count before: 480608
Data count after: 480608
done!
------------------------------
Re-index mapping dumped into ...
user: userid_mapping.csv
item: itemid_mapping.csv
done!
------------------------------
==== Final Data Info: ====
Data Year Range: (2006, 2008)
Rating Threshold: 4.0
Num of interactions: 480608
Num of distinct users: 2103
Num of distinct items: 9519


,userID,movieID,rating,date_day,date_month,date_year,date_hour,date_minute,date_second,timestamp,label
0,0,2,1.0,29,10,2006,23,17,16,2006-10-29 23:17:16,0
1,0,31,4.5,29,10,2006,23,23,44,2006-10-29 23:23:44,1
2,0,98,4.0,29,10,2006,23,30,8,2006-10-29 23:30:08,1
3,0,141,2.0,29,10,2006,23,16,52,2006-10-29 23:16:52,0
4,0,144,4.0,29,10,2006,23,29,30,2006-10-29 23:29:30,1


### Preparing Train/Valid/Test Set

In [3]:
train_df, valid_df, test_df = DataPreprocessor().stratified_time_split(
    interaction_df,
    time_col="timestamp",
    train_ratio=0.75,
    val_ratio=0.1,
    test_ratio=0.15,
)

Splitting data into train/valid/test by time period with ratio=(0.75 : 0.1 : 0.15):
train: 359669 (74.84%
valid: 47149 (9.81%)
test: 73790 (15.35%)
------------------------------ 

Check target label distribution after splitting (%):
train label
0    0.556042
1    0.443958
Name: proportion, dtype: float64
valid label
0    0.611042
1    0.388958
Name: proportion, dtype: float64
test label
0    0.585526
1    0.414474
Name: proportion, dtype: float64


In [4]:
# NOTE: At training, we use interaction graph from train_df for train and validation
train_graph = DataPreprocessor().create_interaction_graph(train_df)

# NOTE: At inference, we can use graph of (train_df + valid_df)
# train_valid_graph = utils.create_interaction_graph(pd.concat([train_df, valid_df], axis=0))

Creating interaction graph...
Drop negative samples
  Num of all interactions: 359669
  Num of positive interactions: 159678 

Building edges...
Building bi-directed edges (duplications)...
Building labels...
Building bi-directed labels (duplications)... 

Interaction Graph: Data(edge_index=[2, 319356], edge_label=[319356])
Edge Index: tensor([[   0,    0,    0,  ..., 2953,  776, 3094],
        [1147, 1234,  702,  ..., 2102, 2102, 2102]])


In [19]:
print(test_df["userID"].nunique())
print(test_df["movieID"].nunique())
test_df.head()
tmp_df = test_df.groupby("userID").agg(pos_cnt=("label", "sum"), all_cnt=("label", "count")).sort_values(
    "pos_cnt", ascending=False
)
tmp_df["neg_cnt"] = tmp_df["all_cnt"] - tmp_df["pos_cnt"]
tmp_df.sort_values(
    "all_cnt", ascending=False
).head(10)


2103
7626


,pos_cnt,all_cnt,neg_cnt
userID,,,
908,27,437,410
945,156,352,196
1966,67,350,283
1232,150,319,169
448,64,310,246
862,122,301,179
181,145,260,115
751,1,256,255
186,83,251,168


In [5]:
train_dataset = UserItemPairDataset(train_df)
valid_dataset = UserItemPairDataset(valid_df)
test_dataset = UserItemPairDataset(test_df)
print("train data count:", len(train_dataset))
print("valid data count:", len(valid_dataset))
print("test data count:", len(test_dataset))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)


train data count: 359669
valid data count: 47149
test data count: 73790


In [12]:
from common.mmgcn_v2 import GCN_ID

class GCNRecLightning(LightningModule):
    def __init__(self, edge_index, num_user, num_item, dim_id=64, lr=1e-3):
        super().__init__()
        self.save_hyperparameters()

        self.model = GCN_ID(
            edge_index=edge_index,
            num_user=num_user,
            num_item=num_item,
            dim_id=dim_id,
            aggr_mode="mean",
            concate=True,
        )

        self.num_user = num_user
        self.num_item = num_item
        self.lr = lr
        self.emb = None

        self.val_step_outputs = []
        self.test_step_outputs = []
        self.val_results = dict()
        self.test_results = dict()


    def forward(self):
        return self.model().to(self.device)

    def _compute_scores(self, emb, user_idx, item_idx):
        # NOTE: the dimension of emb = [(num_user+num_item), emb_dim]
        # return (emb[user_idx] * emb[self.num_user + item_idx]).sum(dim=1)
        return torch.sigmoid((emb[user_idx] * emb[self.num_user + item_idx]).sum(dim=1))

    def _evaluate(self, scores, label):
        # loss = F.binary_cross_entropy_with_logits(scores, label)
        # preds = torch.sigmoid(scores) > 0.5
        loss = F.binary_cross_entropy(scores, label)
        preds = scores > 0.5
        acc = accuracy_score(label.cpu(), preds.cpu())
        prec = precision_score(label.cpu(), preds.cpu())
        rec = recall_score(label.cpu(), preds.cpu())
        f1 = f1_score(label.cpu(), preds.cpu())

        return loss, acc, prec, rec, f1

    def training_step(self, batch, batch_idx):
        user, item, label = batch
        emb = self.forward()
        scores = self._compute_scores(emb, user, item)
        loss, acc, prec, rec, f1 = self._evaluate(scores, label)
        self.log_dict({'train_loss': loss, 'train_acc': acc, 'train_prec': prec, 'train_rec': rec, 'train_f1': f1})
        return loss
    
    def on_validation_epoch_start(self):
        self.emb = self.forward()  # Compute embedding once for val phase

    def validation_step(self, batch, batch_idx):
        user, item, label = batch
        scores = self._compute_scores(self.emb, user, item)

        self.val_step_outputs.append({
            "users": user,
            "items": item,
            "scores": scores,
            "labels": label,
        })
    
    def on_validation_epoch_end(self):
        outputs = self.val_step_outputs
        all_users = torch.cat([x["users"] for x in outputs])
        all_items = torch.cat([x["items"] for x in outputs])
        all_scores = torch.cat([x["scores"] for x in outputs])
        all_labels = torch.cat([x["labels"] for x in outputs])

        loss, acc, prec, rec, f1 = self._evaluate(all_scores, all_labels)
        metrics = {'val_loss': loss, 'val_acc': acc, 'val_prec': prec, 'val_rec': rec, 'val_f1': f1}
        self.log_dict(metrics, prog_bar=True)

        self.val_results = {
            "user": all_users,
            "item": all_items,
            "score": all_scores,
            "label": all_labels,
            "metric": metrics,
            "emb": self.emb,
        }

    # Testing
    def on_test_epoch_start(self):
        self.emb = self.forward()

    def test_step(self, batch, batch_idx):
        user, item, label = batch
        scores = self._compute_scores(self.emb, user, item)

        self.test_step_outputs.append({
            "users": user,
            "items": item,
            "scores": scores,
            "labels": label,
        })


    def on_test_epoch_end(self):
        outputs = self.test_step_outputs
        all_users = torch.cat([x["users"] for x in outputs])
        all_items = torch.cat([x["items"] for x in outputs])
        all_scores = torch.cat([x["scores"] for x in outputs])
        all_labels = torch.cat([x["labels"] for x in outputs])

        loss, acc, prec, rec, f1 = self._evaluate(all_scores, all_labels)
        metrics = {'test_loss': loss, 'test_acc': acc, 'test_prec': prec, 'test_rec': rec, 'test_f1': f1}
        self.log_dict(metrics, prog_bar=True)

        self.test_results = {
            "user": all_users,
            "item": all_items,
            "score": all_scores,
            "label": all_labels,
            "metric": metrics,
            "emb": self.emb,
        }

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.lr)

In [13]:
# setup MLflow logger and callbacks
from pytorch_lightning.loggers import MLFlowLogger
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping, Timer
RUN_NAME = "gcn-baseline-test9"

mlflow_logger = MLFlowLogger(
    experiment_name="gcn-bce-exp",
    run_name=RUN_NAME,
    tracking_uri=MLFLOW_SERVICE_URI,  # can also use http://... for remote
)

checkpoint_callback = ModelCheckpoint(
    monitor="val_f1",  # or "val_f1"
    mode="max",           # or "max" if you're monitoring accuracy/F1
    save_top_k=1,
    save_weights_only=True,
    dirpath="test_checkpoints/",
    filename=f"{RUN_NAME}-best-checkpoint-{{epoch:02d}}-{{val_f1:.2f}}",
    verbose=True
)

early_stopping = EarlyStopping(
    monitor="val_f1",
    patience=3,
    mode="max",
    verbose=True
)

timer = Timer()


In [14]:
trainer = Trainer(
    max_epochs=20,
    logger=mlflow_logger,
    log_every_n_steps=1,
    callbacks=[
        checkpoint_callback,
        early_stopping,
        timer,
    ],
    accelerator='gpu',  # or 'auto', 'gpu'
    devices=[0], # if gpu is available
)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [15]:
model = GCNRecLightning(
    edge_index=train_graph.edge_index,  # shape [2, num_edges]
    num_user=NUM_USER,
    num_item=NUM_ITEM,
    dim_id=64,
    lr=1e-3,
)

trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=valid_loader)


You are using a CUDA device ('NVIDIA GeForce RTX 4070 SUPER') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name  | Type   | Params | Mode 
-----------------------------------------
0 | model | GCN_ID | 793 K  | train
-----------------------------------------
793 K     Trainable params
0         Non-trainable params
793 K     Total params
3.173     Total estimated model params size (MB)
13        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Epoch 3, global step 1332: 'val_f1' was not in top 1


Validation: |          | 0/? [00:00<?, ?it/s]

Monitored metric val_f1 did not improve in the last 3 records. Best score: 0.614. Signaling Trainer to stop.
Epoch 4, global step 1665: 'val_f1' was not in top 1


🏃 View run gcn-baseline-test9 at: http://140.112.106.216:3683/#/experiments/2/runs/e80a43504c5c45ec832d4418a94eecbf
🧪 View experiment at: http://140.112.106.216:3683/#/experiments/2


### Inference

In [16]:
# If you know the file path
best_model_path = "test_checkpoints/gcn-baseline-test8-best-checkpoint-epoch=09-val_f1=0.57.ckpt"
model = GCNRecLightning.load_from_checkpoint(
    checkpoint_path=best_model_path, 
    edge_index=train_graph.edge_index,
    num_user=NUM_USER,
    num_item=NUM_ITEM,
    dim_id=64,
    lr=1e-3,
)


In [17]:
trainer.test(model=model, dataloaders=test_loader)


/Users/jefferybai/Desktop/Master/BILAB/Master Thesis/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Testing: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.47246167063713074    │
│          test_f1          │    0.5486851930618286     │
│         test_loss         │     4.223628520965576     │
│         test_prec         │    0.44704103469848633    │
│         test_rec          │    0.7101534008979797     │
└───────────────────────────┴───────────────────────────┘

🏃 View run gcn-baseline-test8 at: http://140.112.106.216:3683/#/experiments/2/runs/446164e67b3a4e1cab52e9d103657696
🧪 View experiment at: http://140.112.106.216:3683/#/experiments/2


[{'test_loss': 4.223628520965576,
  'test_acc': 0.47246167063713074,
  'test_prec': 0.44704103469848633,
  'test_rec': 0.7101534008979797,
  'test_f1': 0.5486851930618286}]

In [95]:
print(model.test_results["metric"])
model.test_results["emb"].size()


{'test_loss': tensor(2.0647), 'test_acc': 0.47226335272342673, 'test_prec': 0.45361455482208984, 'test_rec': 0.824862396065113, 'test_f1': 0.5853364635489166}


torch.Size([11622, 64])